# Building Complex Deep Learning Models Using Keras Functional API

Run all cells in a fresh Python 3.11–3.13 kernel. CPU only; no API key or GPU is required. All 70,000 real Fashion-MNIST examples are used with the original first-5,000 validation split: 55,000 training, 5,000 validation, 10,000 test. The four original architectures train for 30, 30, 20 and 20 epochs with SGD and batch size 32. Nothing is subsampled. Dataset downloads require internet on first use.

Install the system Graphviz executable before running: `brew install graphviz` on macOS, or `apt-get install graphviz` on Debian/Ubuntu/Colab. Python `pydot` alone is not the Graphviz executable. Restart the kernel after installing packages if older TensorFlow/Keras versions were already imported.

Maintenance: define the source's missing class names; remove repeated assignment; retain the actual ten-class softmax heads; explicitly map both targets and both accuracy metrics for Keras 3. Seed initialization/shuffling and use float32, bounded-thread tf.data batches and named multi-output dictionaries. Preserve full data and epochs. No architecture, training or benchmark result is invented. The source's historical figures/results are distinct from this fresh run. All 18 complete immutable gist originals are retained in the nonexecuting appendix.

In [ ]:
import subprocess
import sys
subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check",
               "tensorflow==2.20.0", "keras==3.15.1", "numpy==2.4.6",
               "matplotlib==3.10.9", "pandas==2.3.3", "pydot==4.0.1",
               "pillow==13.0.0", "certifi==2026.10.23"], check=True)
import os
import shutil
import certifi
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["KERAS_BACKEND"] = "tensorflow"
assert shutil.which("dot"), "Install Graphviz: brew install graphviz, or apt-get install graphviz"
import tensorflow as tf
tf.config.threading.set_inter_op_parallelism_threads(1)
tf.config.threading.set_intra_op_parallelism_threads(1)
tf.config.experimental.enable_op_determinism()
print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("Graphviz:", subprocess.check_output(["dot", "-V"], stderr=subprocess.STDOUT, text=True).strip())

## 1. Loading the Data & Building a Simple Model

In [ ]:
from tensorflow import keras
from tensorflow.keras.utils import plot_model
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image
keras.utils.set_random_seed(42)
class_names = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
               "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
models, histories, evaluations = {}, {}, {}

def dataset(inputs, targets, training=False, batch_size=32):
    ds = tf.data.Dataset.from_tensor_slices((inputs, targets))
    if training:
        ds = ds.shuffle(55000, seed=42, reshuffle_each_iteration=True)
    options = tf.data.Options()
    options.threading.private_threadpool_size = 1
    options.threading.max_intra_op_parallelism = 1
    return ds.batch(batch_size).with_options(options).prefetch(1)

def learning_curve(history, title):
    table = pd.DataFrame(history.history)
    print(table.round(6).to_string(index=True))
    table.plot(figsize=(8, 5))
    plt.title(title)
    plt.xlabel("Epoch (zero-based)")
    plt.grid(True)
    plt.gca().set_ylim(0, 1)
    plt.tight_layout()
    plt.show()

def architecture(model, filename):
    plot_model(model, filename, show_shapes=True)
    display(Image(filename=filename))
    print("Input shapes:", model.input_shape, "| output shape:", model.output_shape)
    print("Trainable parameters:", model.count_params())

print("Keras:", keras.__version__, "| NumPy:", np.__version__)
print("Fashion-MNIST classes:", class_names)

In [ ]:
fashion_mnist = keras.datasets.fashion_mnist
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()
X_valid = X_train_full[:5000].astype("float32") / 255.0
X_train = X_train_full[5000:].astype("float32") / 255.0
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]
X_test = X_test.astype("float32") / 255.0
assert X_train_full.shape == (60000, 28, 28) and X_test.shape == (10000, 28, 28)
print("Loaded all 70,000 Fashion-MNIST images; normalized dtype:", X_train.dtype)

In [ ]:
print('The size of the training data is {} and the size of the train labels is {}' .format(X_train.shape , y_train.shape))
print('The size of the validation data is {} and the size of the validation labels is {}' .format(X_valid.shape , y_valid.shape))
print('The size of the test data is {} and the size of the test labels is {}' .format(X_test.shape , y_test.shape))

In [ ]:
n_rows = 4
n_cols = 10
plt.figure(figsize=(n_cols * 1.2, n_rows * 1.2))
for row in range(n_rows):
    for col in range(n_cols):
        index = n_cols * row + col
        plt.subplot(n_rows, n_cols, index + 1)
        plt.imshow(X_train[index], cmap="binary", interpolation="nearest")
        plt.axis('off')
        plt.title(class_names[y_train[index]], fontsize=12)
plt.subplots_adjust(wspace=0.2, hspace=0.5)
plt.show()
print("Displayed the first 40 training images with their actual class labels.")

In [ ]:
X_train = X_train.reshape(X_train.shape[0], 28*28)
X_valid = X_valid.reshape(X_valid.shape[0], 28*28)
X_test = X_test.reshape(X_test.shape[0], 28*28)
print("Flattened shapes:", X_train.shape, X_valid.shape, X_test.shape)

In [ ]:
keras.utils.set_random_seed(42)
input_ = keras.layers.Input(shape=[28*28])
hidden1 = keras.layers.Dense(30, activation="relu")(input_)
hidden2 = keras.layers.Dense(30, activation="relu")(hidden1)
output = keras.layers.Dense(10, activation="softmax", name="output")(hidden2)
model = keras.models.Model(inputs=input_, outputs=output)
models["simple_mlp"] = model
architecture(model, "simple_mlp.png")

In [ ]:
model.compile(loss="sparse_categorical_crossentropy", optimizer="sgd", metrics=["accuracy"])
history = model.fit(dataset(X_train, y_train, training=True), epochs=30,
                    validation_data=dataset(X_valid, y_valid), verbose=0, shuffle=False)
histories["simple_mlp"] = history
print("Completed 30 epochs on all 55,000 training and 5,000 validation examples.")
learning_curve(history, "Simple Mlp")

In [ ]:
test = model.evaluate(dataset(X_test, y_test, batch_size=100), verbose=0, return_dict=True)
evaluations["simple_mlp"] = test
print("Test examples:", len(y_test))
for metric, value in sorted(test.items()):
    print(f"{metric}: {value:.6f}")

## 2. Building a Wide & Deep Neural Network

In [ ]:
keras.utils.set_random_seed(42)
input_ = keras.layers.Input(shape=[28*28])
hidden1 = keras.layers.Dense(30, activation="relu")(input_)
hidden2 = keras.layers.Dense(30, activation="relu")(hidden1)
concat = keras.layers.concatenate([input_, hidden2])
output = keras.layers.Dense(10, activation="softmax", name="output")(concat)
model = keras.models.Model(inputs=input_, outputs=output)
models["wide_deep"] = model
architecture(model, "wide_deep.png")

In [ ]:
model.compile(loss="sparse_categorical_crossentropy", optimizer="sgd", metrics=["accuracy"])
history = model.fit(dataset(X_train, y_train, training=True), epochs=30,
                    validation_data=dataset(X_valid, y_valid), verbose=0, shuffle=False)
histories["wide_deep"] = history
print("Completed 30 epochs on all 55,000 training and 5,000 validation examples.")
learning_curve(history, "Wide Deep")

In [ ]:
test = model.evaluate(dataset(X_test, y_test, batch_size=100), verbose=0, return_dict=True)
evaluations["wide_deep"] = test
print("Test examples:", len(y_test))
for metric, value in sorted(test.items()):
    print(f"{metric}: {value:.6f}")

## 3. Building Wide & Deep Neural Network With Multiple Inputs

In [ ]:
keras.utils.set_random_seed(42)
input_A = keras.layers.Input(shape=[392], name="wide_input")
input_B = keras.layers.Input(shape=[392], name="deep_input")
hidden1 = keras.layers.Dense(30, activation="relu")(input_B)
hidden2 = keras.layers.Dense(30, activation="relu")(hidden1)
concat = keras.layers.concatenate([input_A, hidden2])
output = keras.layers.Dense(10, activation="softmax", name="output")(concat)
model = keras.models.Model(inputs=[input_A, input_B], outputs=output)
models["multiple_inputs"] = model
architecture(model, "multiple_inputs.png")

In [ ]:
model.compile(loss="sparse_categorical_crossentropy", optimizer="sgd", metrics=["accuracy"])
X_train_A, X_train_B = X_train[:, :392], X_train[:, 392:]
X_valid_A, X_valid_B = X_valid[:, :392], X_valid[:, 392:]
X_test_A, X_test_B = X_test[:, :392], X_test[:, 392:]
history = model.fit(dataset((X_train_A, X_train_B), y_train, training=True), epochs=20,
                    validation_data=dataset((X_valid_A, X_valid_B), y_valid), verbose=0, shuffle=False)
histories["multiple_inputs"] = history
print("Completed 20 epochs on all 55,000 training and 5,000 validation examples.")
learning_curve(history, "Multiple Inputs")

In [ ]:
test = model.evaluate(dataset((X_test_A, X_test_B), y_test, batch_size=100), verbose=0, return_dict=True)
evaluations["multiple_inputs"] = test
print("Test examples:", len(y_test))
for metric, value in sorted(test.items()):
    print(f"{metric}: {value:.6f}")

## 4. Building Wide & Deep Neural Network With Multiple Inputs & Outputs

In [ ]:
keras.utils.set_random_seed(42)
input_A = keras.layers.Input(shape=[392], name="wide_input")
input_B = keras.layers.Input(shape=[392], name="deep_input")

hidden1 = keras.layers.Dense(30, activation="relu")(input_B)
hidden2 = keras.layers.Dense(30, activation="relu")(hidden1)

concat = keras.layers.concatenate([input_A, hidden2])

output = keras.layers.Dense(10, activation="softmax",name="main_output")(concat)
aux_output = keras.layers.Dense(10, activation="softmax", name="aux_output")(hidden2)

model = keras.models.Model(inputs=[input_A, input_B],
                           outputs={"main_output": output, "aux_output": aux_output})
models["multiple_outputs"] = model
architecture(model, "multiple_outputs.png")

In [ ]:
model.compile(
    loss={"main_output": "sparse_categorical_crossentropy", "aux_output": "sparse_categorical_crossentropy"},
    optimizer="sgd",
    loss_weights={"main_output": 0.7, "aux_output": 0.3},
    metrics={"main_output": ["accuracy"], "aux_output": ["accuracy"]},
)
print("Compiled both softmax heads with sparse categorical cross-entropy and accuracy.")
print("Loss weights: main_output=0.7, aux_output=0.3")

In [ ]:
X_train_A, X_train_B = X_train[:, :392], X_train[:, 392:]
X_valid_A, X_valid_B = X_valid[:, :392], X_valid[:, 392:]
X_test_A, X_test_B = X_test[:, :392], X_test[:, 392:]
train_targets = {"main_output": y_train, "aux_output": y_train}
valid_targets = {"main_output": y_valid, "aux_output": y_valid}
history = model.fit(dataset((X_train_A, X_train_B), train_targets, training=True), epochs=20,
                    validation_data=dataset((X_valid_A, X_valid_B), valid_targets), verbose=0, shuffle=False)
histories["multiple_outputs"] = history
print("Completed 20 epochs; both heads receive all 55,000 training labels and 5,000 validation labels.")
learning_curve(history, "Multiple inputs and outputs")

In [ ]:
test_targets = {"main_output": y_test, "aux_output": y_test}
test = model.evaluate(dataset((X_test_A, X_test_B), test_targets, batch_size=100), verbose=0, return_dict=True)
evaluations["multiple_outputs"] = test
print("Test examples:", len(y_test))
for metric in ["loss", "main_output_loss", "aux_output_loss", "main_output_accuracy", "aux_output_accuracy"]:
    print(f"{metric}: {test[metric]:.6f}")

## Independent full-test-set regression checks

Recompute accuracy and cross-entropy from all 10,000 predictions with NumPy rather than trusting printed Keras metrics. Check tensor dimensions, normalized pixel values, graph parameter counts, finite training histories, full epoch/batch counts, both probability heads and 0.7/0.3 weighted loss.

In [ ]:
assert X_train.shape == (55000, 784) and X_valid.shape == (5000, 784) and X_test.shape == (10000, 784)
assert np.array_equal(X_train.reshape(-1, 28, 28) * 255.0, X_train_full[5000:].astype("float32"))
assert np.array_equal(y_valid, y_train_full[:5000]) and np.array_equal(y_train, y_train_full[5000:])
for x, y in [(X_train, y_train), (X_valid, y_valid), (X_test, y_test)]:
    assert x.dtype == np.float32 and np.isfinite(x).all() and x.min() >= 0 and x.max() <= 1
    assert np.array_equal(np.unique(y), np.arange(10))
assert np.array_equal(np.concatenate([X_test_A, X_test_B], axis=1), X_test)
expected_params = {"simple_mlp": 24790, "wide_deep": 32630, "multiple_inputs": 16950, "multiple_outputs": 17260}
for name, trained in models.items():
    expected_epochs = 30 if name in ["simple_mlp", "wide_deep"] else 20
    assert len(histories[name].epoch) == expected_epochs
    assert all(len(v) == expected_epochs and np.isfinite(v).all() for v in histories[name].history.values())
    assert int(trained.optimizer.iterations.numpy()) == expected_epochs * int(np.ceil(55000 / 32))
    assert trained.count_params() == expected_params[name]
    inputs = (X_test_A, X_test_B) if name in ["multiple_inputs", "multiple_outputs"] else X_test
    predictions = trained.predict(dataset(inputs, y_test), verbose=0)
    predictions = predictions if isinstance(predictions, dict) else {"output": predictions}
    independent_losses = {}
    for head, probabilities in predictions.items():
        assert probabilities.shape == (10000, 10) and np.isfinite(probabilities).all()
        np.testing.assert_allclose(probabilities.sum(axis=1), 1, atol=1e-6)
        accuracy = float(np.mean(probabilities.argmax(axis=1) == y_test))
        loss = float(-np.log(np.clip(probabilities[np.arange(10000), y_test], np.finfo(np.float32).tiny, 1)).mean())
        key = head + "_" if name == "multiple_outputs" else ""
        np.testing.assert_allclose(accuracy, evaluations[name][key + "accuracy"], atol=1e-6)
        np.testing.assert_allclose(loss, evaluations[name][key + "loss"], atol=2e-6)
        independent_losses[head] = loss
    if name == "multiple_outputs":
        np.testing.assert_allclose(0.7 * independent_losses["main_output"] + 0.3 * independent_losses["aux_output"], evaluations[name]["loss"], atol=2e-6)
    print(name, ":", expected_epochs, "epochs; parameters:", trained.count_params(), "; full-test NumPy checks passed.")
print("All four architectures, 100 total epochs and all 10,000 test predictions per model verified.")

## Immutable original source gists

Historical originals, retained verbatim for provenance; do not execute these Markdown blocks.

### Original example 1

https://gist.github.com/youssefHosni/86b58650387c82fde3750e660768d109/98e2beff9ad3a1c8f082bf83780b7554ace23e79

gistfile1.txt

```python
from tensorflow import keras
from tensorflow.keras.utils import plot_model
import pandas as pd
import matplotlib.pyplot as plt
```

### Original example 2

https://gist.github.com/youssefHosni/b67831a97786aa53236a3493e059c272/161aabcef726b9a42001807dd0be99d0406e34f9

gistfile1.txt

```python
fashion_mnist = keras.datasets.fashion_mnist
(X_train_full, y_train_full), (X_test, y_test) = fashion_mnist.load_data()

X_valid, X_train = X_train_full[:5000] / 255., X_train_full[5000:] / 255.
y_valid, y_train = y_train_full[:5000], y_train_full[5000:]
X_test = X_test / 255.
```

### Original example 3

https://gist.github.com/youssefHosni/ca74b68a15bd3c83ea9585e3bcc343dc/d55deca10fb67856900eff90d16f80bd493c21cc

gistfile1.txt

```python
print('The size of the trainig data is {} and the size of the train labels is {}' .format(X_train.shape , y_train.shape))
print('The size of the validation data is {} and the size of the validation labels is {}' .format(X_valid.shape , y_valid.shape))
print('The size of the test data is {} and the size of the test labels is {}' .format(X_test.shape , y_test.shape))
```

### Original example 4

https://gist.github.com/youssefHosni/c123e1671c2aef63e8a95977c8c7c8b8/7e838bfd770b67cf1adf4d90ce73d0d2675c7f7c

gistfile1.txt

```python
n_rows = 4
n_cols = 10
plt.figure(figsize=(n_cols * 1.2, n_rows * 1.2))
for row in range(n_rows):
    for col in range(n_cols):
        index = n_cols * row + col
        plt.subplot(n_rows, n_cols, index + 1)
        plt.imshow(X_train[index], cmap="binary", interpolation="nearest")
        plt.axis('off')
        plt.title(class_names[y_train[index]], fontsize=12)
plt.subplots_adjust(wspace=0.2, hspace=0.5)
plt.show()
```

### Original example 5

https://gist.github.com/youssefHosni/264c5f7ee232da151e0cba1c484f4bc0/d3f0e04aba6a5899c6751700f3a3d7bb9c27eb58

gistfile1.txt

```python
X_train = X_train.reshape(X_train.shape[0], 28*28)
X_valid = X_valid.reshape(X_valid.shape[0], 28*28)
X_test = X_test.reshape(X_test.shape[0], 28*28)
```

### Original example 6

https://gist.github.com/youssefHosni/fb8b05c2097a0f6d72f8534a7d3ea403/680743a7bd8def2bf2db36183c1fb05666aab347

gistfile1.txt

```python
input_ = keras.layers.Input(shape=[28*28])
hidden1 = keras.layers.Dense(30, activation="relu")(input_)
hidden2 = keras.layers.Dense(30, activation="relu")(hidden1)
output = output = keras.layers.Dense(10, activation="softmax", name="output")(hidden2)
model = keras.models.Model(inputs=[input_], outputs=[output])

keras.utils.plot_model(model, "Neural_Network.png", show_shapes=True)
```

### Original example 7

https://gist.github.com/youssefHosni/28377ca0b62bb04aa1d3c2d90039b8eb/27f1d840e318988d4443f3fb395c2e8fccb2a10b

gistfile1.txt

```python
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

history = model.fit(X_train, y_train, epochs=30, validation_data=(X_valid, y_valid))

pd.DataFrame(history.history).plot(figsize=(8, 5))
plt.grid(True)
plt.gca().set_ylim(0, 1)
plt.show()
```

### Original example 8

https://gist.github.com/youssefHosni/36c6d826b3a54af199363010969152e0/e7193e51752e132354e7343ab9c1a232b5fc8856

gistfile1.txt

```python
model.evaluate(X_test, y_test)
```

### Original example 9

https://gist.github.com/youssefHosni/51ad34cb12e043b570fa0da991cc861d/a7a39fea1c8d6b7f6fcae53ef61343b6ca15ab2d

gistfile1.txt

```python
input_ = keras.layers.Input(shape=[28*28])
hidden1 = keras.layers.Dense(30, activation="relu")(input_)
hidden2 = keras.layers.Dense(30, activation="relu")(hidden1)
concat = keras.layers.concatenate([input_, hidden2])
output = output = keras.layers.Dense(10, activation="softmax", name="output")(concat)
model = keras.models.Model(inputs=[input_], outputs=[output])

keras.utils.plot_model(model, "Deep & wide.png", show_shapes=True)
```

### Original example 10

https://gist.github.com/youssefHosni/e11ac712760df630bda8bb9cfe43332e/185d77ccaa7d0335f9f7d8245cdb2504be9abf78

gistfile1.txt

```python
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

history = model.fit(X_train, y_train, epochs=30, validation_data=(X_valid, y_valid))

pd.DataFrame(history.history).plot(figsize=(8, 5))
plt.grid(True)
plt.gca().set_ylim(0, 1)
plt.show()
```

### Original example 11

https://gist.github.com/youssefHosni/262bf4a953bdd0c95596a3862aad2394/70ea1c0e332e076d8c22a1f808774439d1c781a1

gistfile1.txt

```python
model.evaluate(X_test, y_test)
```

### Original example 12

https://gist.github.com/youssefHosni/c0427f28889b64c590d290c2b38b2b53/dbc5ef88e4a859454955534592937d9d7ffc475c

gistfile1.txt

```python
input_A = keras.layers.Input(shape=[392], name="wide_input")
input_B = keras.layers.Input(shape=[392], name="deep_input")
hidden1 = keras.layers.Dense(30, activation="relu")(input_B)
hidden2 = keras.layers.Dense(30, activation="relu")(hidden1)
concat = keras.layers.concatenate([input_A, hidden2])
output = keras.layers.Dense(10, activation="softmax", name="output")(concat)
model = keras.models.Model(inputs=[input_A, input_B], outputs=[output])

keras.utils.plot_model(model, "multiple_inputs_model.png", show_shapes=True)
```

### Original example 13

https://gist.github.com/youssefHosni/06ca43eb001b6fa8bdaf8508f286162f/7a1b508d44597bbc91a1b1166cb36abe9ac1a223

gistfile1.txt

```python
model.compile(loss="sparse_categorical_crossentropy",
              optimizer="sgd",
              metrics=["accuracy"])

X_train_A, X_train_B = X_train[:, :392], X_train[:, 392:]
X_valid_A, X_valid_B = X_valid[:, :392], X_valid[:, 392:]
X_test_A, X_test_B = X_test[:, :392], X_test[:, 392:]

history = model.fit((X_train_A, X_train_B), y_train, epochs=20,
                    validation_data=((X_valid_A, X_valid_B), y_valid))
```

### Original example 14

https://gist.github.com/youssefHosni/126a56110895a4e18e5f4728f487ded3/0a7a90e20fedfc2ab1c88e906b9f372dd376b6fd

gistfile1.txt

```python
test = model.evaluate((X_test_A, X_test_B), y_test)
```

### Original example 15

https://gist.github.com/youssefHosni/a4028ffac195786c9b2c8729a695cfb5/6746f9bd19779ec2817a092273d746ce1b068d39

gistfile1.txt

```python
input_A = keras.layers.Input(shape=[392], name="wide_input")
input_B = keras.layers.Input(shape=[392], name="deep_input")

hidden1 = keras.layers.Dense(30, activation="relu")(input_B)
hidden2 = keras.layers.Dense(30, activation="relu")(hidden1)

concat = keras.layers.concatenate([input_A, hidden2])

output = keras.layers.Dense(10, activation="softmax",name="main_output")(concat)
aux_output = keras.layers.Dense(10, activation="softmax", name="aux_output")(hidden2)

model = keras.models.Model(inputs=[input_A, input_B],
                           outputs=[output, aux_output])

keras.utils.plot_model(model, "model.png", show_shapes=True)
```

### Original example 16

https://gist.github.com/youssefHosni/ae8875c9ad8e7b6798df725031484b27/05dd37f0328bff07fddf69046af1b7843b4db433

gistfile1.txt

```python
model.compile(loss=["sparse_categorical_crossentropy","sparse_categorical_crossentropy" ],
              optimizer="sgd",
              loss_weights=[0.7, 0.3],
              metrics=["accuracy"])
```

### Original example 17

https://gist.github.com/youssefHosni/8dbfc0421b963ca02ca6fb74533bc974/8fdf921abd81d9a8d0302a2e45ab56fa6d075f9a

gistfile1.txt

```python
X_train_A, X_train_B = X_train[:, :392], X_train[:, 392:]
X_valid_A, X_valid_B = X_valid[:, :392], X_valid[:, 392:]
X_test_A, X_test_B = X_test[:, :392], X_test[:, 392:]

history = model.fit((X_train_A, X_train_B), y_train, epochs=20,
                    validation_data=((X_valid_A, X_valid_B), y_valid))
```

### Original example 18

https://gist.github.com/youssefHosni/b1c5fe9278798f39ba5fd9fde3630bf0/a8f31e0407b42fe40ec792d7a0c4744a97ce0cee

gistfile1.txt

```python
model.evaluate([X_test_A, X_test_B], [y_test, y_test])
```